ABLATION TRAINING ENVIRONMENT

In [12]:
from pathlib import Path
import ast
import json
import random
import hashlib
import gc

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# 1. Reproducibility
SEED = 7

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 2. Locate project root
PROJECT_ROOT = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (
            p / "experiments" / "05_ablation"
            / "data" / "ablation_manifest.csv"
        ).is_file()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate ablation project."
    )

EXPERIMENT_DIR = (
    PROJECT_ROOT / "experiments" / "05_ablation"
)

DATA_DIR = EXPERIMENT_DIR / "data"
RESULTS_DIR = EXPERIMENT_DIR / "results"

MODEL_NOTEBOOK = (
    EXPERIMENT_DIR / "02_ablation_model.ipynb"
)

assert MODEL_NOTEBOOK.is_file()

# 3. Load four validated datasets
DATASET_FILES = {
    "Electricity": "electricity_ablation.pt",
    "Financial": "financial_ablation.pt",
    "CMIP6 Wind": "cmip6_wind_ablation.pt",
    "CMIP6 Pressure": "cmip6_pressure_candidate_2500.pt"
}

ABLATION_DATASETS = {}

for name, filename in DATASET_FILES.items():

    path = DATA_DIR / filename

    bundle = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    assert bundle["train"].ndim == 2
    assert bundle["test"].ndim == 2

    assert (
        bundle["train"].shape[1]
        == bundle["test"].shape[1]
        == bundle["input_dim"]
    )

    assert torch.isfinite(bundle["train"]).all()
    assert torch.isfinite(bundle["test"]).all()

    ABLATION_DATASETS[name] = bundle

# 4. Author's ablation configuration
PATCH_LENGTHS = [70, 80, 90, 100, 110, 120, 130]
HORIZONS = [2, 4, 6, 8, 10, 12, 14, 16]

MODEL_VARIANTS = [
    "DLinear",
    "LSTM",
    "PatchTST",
    "PatchTST+LinearLatent",
    "KoopPatchTST_NoLyap",
    "KoopPatchTST_Full"
]

D_MODEL = 96
D_LATENT = 96

LSTM_HIDDEN_DIM = 96
LSTM_NUM_LAYERS = 2

NUM_ENCODER_LAYERS = 3
NUM_ATTENTION_HEADS = 4
DIM_FEEDFORWARD = 96

RHO_MAX = 0.99
LYAP_WEIGHT = 0.1

EPOCHS = 4000
LEARNING_RATE = 3e-4

# 5. Extract model definitions from File 2
REQUIRED_DEFINITIONS = {
    "_orth",
    "StrictStableKoopmanOperator",
    "LinearLatentPropagator",
    "SinCosPosEnc",
    "PatchEmbed1D",
    "PatchTST_Backbone",
    "PatchTSTBaseline",
    "PatchTST_UnconstrainedLatent",
    "Koopformer_PatchTST",
    "SimpleLSTMForecaster",
    "DLinearForecaster",
    "create_ablation_model"
}

with open(
    MODEL_NOTEBOOK, "r", encoding="utf-8"
) as f:
    model_notebook = json.load(f)

definitions = []
found_names = set()

for cell in model_notebook["cells"]:

    if cell["cell_type"] != "code":
        continue

    source = cell["source"]

    if isinstance(source, list):
        source = "".join(source)

    if not source.strip():
        continue

    tree = ast.parse(source)

    for node in tree.body:

        if isinstance(
            node, (ast.ClassDef, ast.FunctionDef)
        ) and node.name in REQUIRED_DEFINITIONS:

            definitions.append(node)
            found_names.add(node.name)

# 6. Ensure no model definition is missing
missing = REQUIRED_DEFINITIONS - found_names

if missing:
    raise RuntimeError(
        f"Missing model definitions: {sorted(missing)}"
    )

# 7. Execute only the selected definitions
module = ast.Module(
    body=definitions,
    type_ignores=[]
)

ast.fix_missing_locations(module)

exec(
    compile(
        module,
        str(MODEL_NOTEBOOK),
        "exec"
    ),
    globals()
)

# 8. Quick model loading test
test_model, test_koop_attr, test_lyap = (
    create_ablation_model(
        model_name="KoopPatchTST_Full",
        input_dim=5,
        patch_len=70,
        horizon=2
    )
)

test_model.eval()

with torch.no_grad():
    test_output = test_model(
        torch.zeros(2, 70, 5)
    )

assert test_output.shape == (2, 10)
assert test_koop_attr == "koop"
assert test_lyap == 0.1

del test_model

# Reset RNG after import validation
set_seed()

# 9. Dataset summary
dataset_summary = []

for name, bundle in ABLATION_DATASETS.items():

    dataset_summary.append({
        "Dataset": name,
        "Channels": bundle["input_dim"],
        "Train": tuple(bundle["train"].shape),
        "Test": tuple(bundle["test"].shape),
        "Status": bundle["status"]
    })

# 10. Final summary
print("=== ABLATION TRAINING ENVIRONMENT ===")

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)
print("PyTorch version:", torch.__version__)
print("Seed:", SEED)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\n=== DATASET VALIDATION ===")
display(pd.DataFrame(dataset_summary))

print("\n=== MODEL DEFINITIONS ===")
print("Source:", MODEL_NOTEBOOK.name)
print("Definitions loaded:", len(found_names))
print("All required definitions found:",
      found_names == REQUIRED_DEFINITIONS)

print("\n=== TRAINING CONFIGURATION ===")
print("Models:", len(MODEL_VARIANTS))
print("Patch lengths:", len(PATCH_LENGTHS))
print("Horizons:", len(HORIZONS))
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)

total_configs = (
    len(ABLATION_DATASETS)
    * len(PATCH_LENGTHS)
    * len(HORIZONS)
    * len(MODEL_VARIANTS)
)

print("\nTotal training configurations:", total_configs)

print("\nModel import validation passed!")
print("Training environment initialized!")


=== ABLATION TRAINING ENVIRONMENT ===
Project root: c:\Users\Win 11 New\Desktop\pa gk
Device: cuda
PyTorch version: 2.0.1+cu118
Seed: 7
GPU: Quadro T1000

=== DATASET VALIDATION ===


,Dataset,Channels,Train,Test,Status
0,Electricity,6,"(2800, 6)","(700, 6)",verified_local_pipeline
1,Financial,2,"(2320, 2)","(580, 2)",verified_local_pipeline
2,CMIP6 Wind,5,"(2336, 5)","(584, 5)",verified_local_extraction
3,CMIP6 Pressure,5,"(2000, 5)","(500, 5)",candidate_2500



=== MODEL DEFINITIONS ===
Source: 02_ablation_model.ipynb
Definitions loaded: 12
All required definitions found: True

=== TRAINING CONFIGURATION ===
Models: 6
Patch lengths: 7
Horizons: 8
Epochs: 4000
Learning rate: 0.0003

Total training configurations: 1344

Model import validation passed!
Training environment initialized!


UNIFIED TRAIN/TEST SLIDING WINDOWS

In [2]:
# 1. Sliding-window function (author's implementation)
def build_ablation_windows(
    data,
    patch_len: int,
    horizon: int
):
    """
    Input:
        data: (N, F)
        patch_len: number of past timestamps
        horizon: number of future timestamps

    Returns:
        X: (samples, patch_len, F)
        Y: (samples, F * horizon)

    Matches the author's sliding-window convention.
    """

    if isinstance(data, torch.Tensor):
        data = data.detach().cpu().numpy()

    data = np.asarray(data, dtype=np.float32)

    if data.ndim != 2:
        raise ValueError("Expected data shape (N, F).")

    n_windows = len(data) - patch_len - horizon

    if n_windows <= 0:
        raise ValueError(
            f"Insufficient data: N={len(data)}, "
            f"P={patch_len}, H={horizon}"
        )

    X, Y = [], []

    for i in range(n_windows):

        X.append(
            data[i:i + patch_len]
        )

        Y.append(
            data[
                i + patch_len:
                i + patch_len + horizon
            ].T.reshape(-1)
        )

    x_in = torch.tensor(
        np.stack(X),
        dtype=torch.float32
    )

    x_out = torch.tensor(
        np.stack(Y),
        dtype=torch.float32
    )

    return x_in, x_out


# 2. Unified dataset preparation function
def prepare_ablation_data(
    dataset_name: str,
    patch_len: int,
    horizon: int
):
    """
    Prepare Train/Test tensors for one configuration.

    Data are already scaled in File 1.
    Keep tensors on CPU until actual training.
    """

    if dataset_name not in ABLATION_DATASETS:
        raise ValueError(
            f"Unknown dataset: {dataset_name}"
        )

    if patch_len not in PATCH_LENGTHS:
        raise ValueError(
            f"Unsupported patch length: {patch_len}"
        )

    if horizon not in HORIZONS:
        raise ValueError(
            f"Unsupported horizon: {horizon}"
        )

    bundle = ABLATION_DATASETS[dataset_name]

    train = bundle["train"]
    test = bundle["test"]

    # Create windows separately to preserve split boundary
    x_train, y_train = build_ablation_windows(
        train,
        patch_len,
        horizon
    )

    x_test, y_test = build_ablation_windows(
        test,
        patch_len,
        horizon
    )

    # Validate shapes
    F = bundle["input_dim"]

    assert x_train.shape == (
        len(train) - patch_len - horizon,
        patch_len,
        F
    )

    assert y_train.shape == (
        len(x_train),
        F * horizon
    )

    assert x_test.shape == (
        len(test) - patch_len - horizon,
        patch_len,
        F
    )

    assert y_test.shape == (
        len(x_test),
        F * horizon
    )

    # Validate numerical values
    for tensor in [
        x_train, y_train,
        x_test, y_test
    ]:
        assert torch.isfinite(tensor).all()

    return x_train, y_train, x_test, y_test


# 3. Representative configurations
test_configurations = {
    "Electricity": (70, 8),
    "Financial": (80, 4),
    "CMIP6 Wind": (120, 8),
    "CMIP6 Pressure": (130, 16)
}

# 4. Validate all four datasets
window_results = []

for dataset_name, (p, h) in test_configurations.items():

    x_train, y_train, x_test, y_test = (
        prepare_ablation_data(
            dataset_name,
            patch_len=p,
            horizon=h
        )
    )

    bundle = ABLATION_DATASETS[dataset_name]

    train = bundle["train"]
    test = bundle["test"]

    # Check first Train window
    torch.testing.assert_close(
        x_train[0],
        train[:p],
        rtol=0,
        atol=0
    )

    torch.testing.assert_close(
        y_train[0],
        train[p:p + h].T.reshape(-1),
        rtol=0,
        atol=0
    )

    # Check first Test window
    torch.testing.assert_close(
        x_test[0],
        test[:p],
        rtol=0,
        atol=0
    )

    torch.testing.assert_close(
        y_test[0],
        test[p:p + h].T.reshape(-1),
        rtol=0,
        atol=0
    )

    # Check last Train and Test windows
    for series, X, Y in [
        (train, x_train, y_train),
        (test, x_test, y_test)
    ]:

        last_i = len(X) - 1

        torch.testing.assert_close(
            X[-1],
            series[last_i:last_i + p],
            rtol=0,
            atol=0
        )

        torch.testing.assert_close(
            Y[-1],
            series[
                last_i + p:
                last_i + p + h
            ].T.reshape(-1),
            rtol=0,
            atol=0
        )

    window_results.append({
        "Dataset": dataset_name,
        "P": p,
        "H": h,
        "X_train": tuple(x_train.shape),
        "Y_train": tuple(y_train.shape),
        "X_test": tuple(x_test.shape),
        "Y_test": tuple(y_test.shape),
        "Status": "PASSED"
    })


# 5. Summary
window_validation_df = pd.DataFrame(
    window_results
)

assert len(window_validation_df) == 4
assert (window_validation_df["Status"] == "PASSED").all()

print("=== ABLATION TRAIN/TEST WINDOWS ===")

display(window_validation_df)

print("\n=== VALIDATION SUMMARY ===")

print("Datasets tested:", len(window_validation_df))
print("Input format: (samples, patch_len, channels)")
print("Target format: (samples, channels * horizon)")

print("\nTrain/Test windows created separately.")
print("First and last windows validated.")
print("No cross-boundary windows introduced.")
print("All four datasets passed!")

print("\nCell 2 completed successfully!")


=== ABLATION TRAIN/TEST WINDOWS ===


,Dataset,P,H,X_train,Y_train,X_test,Y_test,Status
0,Electricity,70,8,"(2722, 70, 6)","(2722, 48)","(622, 70, 6)","(622, 48)",PASSED
1,Financial,80,4,"(2236, 80, 2)","(2236, 8)","(496, 80, 2)","(496, 8)",PASSED
2,CMIP6 Wind,120,8,"(2208, 120, 5)","(2208, 40)","(456, 120, 5)","(456, 40)",PASSED
3,CMIP6 Pressure,130,16,"(1854, 130, 5)","(1854, 80)","(354, 130, 5)","(354, 80)",PASSED



=== VALIDATION SUMMARY ===
Datasets tested: 4
Input format: (samples, patch_len, channels)
Target format: (samples, channels * horizon)

Train/Test windows created separately.
First and last windows validated.
No cross-boundary windows introduced.
All four datasets passed!

Cell 2 completed successfully!


MSE & LYAPUNOV LOSS VALIDATION

In [3]:
# 1. Loss function following the author's implementation
def compute_ablation_loss(
    model,
    x,
    y,
    lyap_weight=0.0
):
    """
    Returns:
        total_loss
        prediction_loss
        lyapunov_loss
        predictions
    """

    # Forward pass
    pred, z, z_next = model(
        x,
        return_latents=True
    )

    # Prediction loss
    pred_loss = F.mse_loss(pred, y)

    # Lyapunov regularization
    if lyap_weight:

        P = torch.eye(
            z.shape[1],
            device=z.device,
            dtype=z.dtype
        )

        zp = torch.einsum(
            "bi,ij->bj", z, P
        )

        znp = torch.einsum(
            "bi,ij->bj", z_next, P
        )

        lyap_loss = torch.relu(
            (znp * z_next).sum(1)
            - (zp * z).sum(1)
        ).mean()

    else:
        lyap_loss = torch.zeros(
            (), device=x.device, dtype=x.dtype
        )

    total_loss = (
        pred_loss + lyap_weight * lyap_loss
    )

    return total_loss, pred_loss, lyap_loss, pred


# 2. Prepare a small real batch
LOSS_DATASET = "CMIP6 Wind"
LOSS_PATCH_LEN = 70
LOSS_HORIZON = 8
LOSS_BATCH_SIZE = 8

x_train, y_train, _, _ = prepare_ablation_data(
    LOSS_DATASET,
    LOSS_PATCH_LEN,
    LOSS_HORIZON
)

x_batch = x_train[:LOSS_BATCH_SIZE].to(DEVICE)
y_batch = y_train[:LOSS_BATCH_SIZE].to(DEVICE)

input_dim = x_batch.shape[-1]

assert x_batch.shape == (8, 70, 5)
assert y_batch.shape == (8, 40)

# 3. Validate loss and backward pass for all six models
loss_results = []

for model_name in MODEL_VARIANTS:

    model, koop_attr, lyap_weight = (
        create_ablation_model(
            model_name,
            input_dim,
            LOSS_PATCH_LEN,
            LOSS_HORIZON
        )
    )

    model = model.to(DEVICE)
    model.train()
    model.zero_grad()

    total, mse, lyap, pred = compute_ablation_loss(
        model,
        x_batch,
        y_batch,
        lyap_weight
    )

    assert pred.shape == y_batch.shape

    assert torch.isfinite(total).item()
    assert torch.isfinite(mse).item()
    assert torch.isfinite(lyap).item()

    # Check loss composition
    torch.testing.assert_close(
        total,
        mse + lyap_weight * lyap
    )

    if lyap_weight == 0.0:
        assert lyap.item() == 0.0

    # Verify backpropagation
    total.backward()

    grad_count = sum(
        1 for p in model.parameters()
        if p.grad is not None
    )

    assert grad_count > 0

    assert all(
        torch.isfinite(p.grad).all().item()
        for p in model.parameters()
        if p.grad is not None
    )

    loss_results.append({
        "Model": model_name,
        "Lambda": lyap_weight,
        "MSE": mse.item(),
        "Lyapunov": lyap.item(),
        "Total": total.item(),
        "Gradients": grad_count,
        "Status": "PASSED"
    })

    del model, total, mse, lyap, pred


# 4. Controlled comparison: same initial weights
no_lyap_model, _, no_lyap_weight = (
    create_ablation_model(
        "KoopPatchTST_NoLyap",
        input_dim,
        LOSS_PATCH_LEN,
        LOSS_HORIZON
    )
)

full_model, _, full_weight = (
    create_ablation_model(
        "KoopPatchTST_Full",
        input_dim,
        LOSS_PATCH_LEN,
        LOSS_HORIZON
    )
)

# Force identical parameters
full_model.load_state_dict(
    no_lyap_model.state_dict()
)

no_lyap_model = no_lyap_model.to(DEVICE)
full_model = full_model.to(DEVICE)

# Disable dropout for deterministic comparison
no_lyap_model.eval()
full_model.eval()

no_total, no_mse, no_lyap, no_pred = (
    compute_ablation_loss(
        no_lyap_model,
        x_batch,
        y_batch,
        no_lyap_weight
    )
)

full_total, full_mse, full_lyap, full_pred = (
    compute_ablation_loss(
        full_model,
        x_batch,
        y_batch,
        full_weight
    )
)

# 5. Verify identical predictions and MSE
torch.testing.assert_close(
    no_pred, full_pred,
    rtol=0, atol=0
)

torch.testing.assert_close(
    no_mse, full_mse,
    rtol=0, atol=0
)

# Verify total loss calculation
torch.testing.assert_close(
    full_total,
    full_mse + full_weight * full_lyap
)

# 6. Measure Lyapunov gradient
lyap_grad = torch.autograd.grad(
    full_lyap,
    full_model.koop.S_raw,
    retain_graph=True,
    allow_unused=True
)[0]

lyap_grad_norm = (
    lyap_grad.norm().item()
    if lyap_grad is not None
    else 0.0
)

# 7. Check Koopman singular values
with torch.no_grad():
    z_sample = full_model.backbone(x_batch)

    _, K, sigma = full_model.koop(z_sample)

    max_singular_value = (
        torch.linalg.svdvals(K).max().item()
    )

assert max_singular_value < RHO_MAX + 1e-5

# 8. Final summary
loss_df = pd.DataFrame(loss_results)

assert len(loss_df) == 6
assert (loss_df["Status"] == "PASSED").all()

print("=== ABLATION LOSS VALIDATION ===")
display(loss_df)

print("\n=== CONTROLLED KOOPMAN COMPARISON ===")

print("NoLyap MSE:", no_mse.item())
print("Full MSE:", full_mse.item())

print("NoLyap total loss:", no_total.item())
print("Full total loss:", full_total.item())

print("\nFull Lyapunov loss:", full_lyap.item())
print("Lyapunov gradient norm:", lyap_grad_norm)

print("Maximum singular value:", max_singular_value)
print("Stability bound:", RHO_MAX)

print("\nMSE implementation validated.")
print("Lyapunov loss implementation validated.")
print("All six models passed backward validation.")
print("Cell 3 completed successfully!")


=== ABLATION LOSS VALIDATION ===


,Model,Lambda,MSE,Lyapunov,Total,Gradients,Status
0,DLinear,0.0,0.158238,0.0,0.158238,10,PASSED
1,LSTM,0.0,0.077361,0.0,0.077361,10,PASSED
2,PatchTST,0.0,0.466043,0.0,0.466043,41,PASSED
3,PatchTST+LinearLatent,0.0,0.148720,0.0,0.148720,42,PASSED
4,KoopPatchTST_NoLyap,0.0,0.155046,0.0,0.155046,44,PASSED
5,KoopPatchTST_Full,0.1,0.190148,0.0,0.190148,44,PASSED



=== CONTROLLED KOOPMAN COMPARISON ===
NoLyap MSE: 0.18608470261096954
Full MSE: 0.18608470261096954
NoLyap total loss: 0.18608470261096954
Full total loss: 0.18608470261096954

Full Lyapunov loss: 0.0
Lyapunov gradient norm: 0.0
Maximum singular value: 0.9186217188835144
Stability bound: 0.99

MSE implementation validated.
Lyapunov loss implementation validated.
All six models passed backward validation.
Cell 3 completed successfully!


UNIFIED ABLATION TRAINING LOOP

In [4]:
def train_ablation_model(
    model,
    x_train,
    y_train,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    koop_attr=None,
    lyap_weight=0.0,
    device=DEVICE,
    optimizer=None,
    start_epoch=0,
    history=None,
    checkpoint_callback=None,
    checkpoint_interval=500,
    verbose=True
):
    """
    Full-batch training following the author's implementation.

    Optional arguments support checkpoint/resume
    without changing the optimization procedure.
    """

    # 1. Validate inputs
    assert x_train.ndim == 3
    assert y_train.ndim == 2
    assert len(x_train) == len(y_train)
    assert 0 <= start_epoch < epochs

    if start_epoch > 0 and optimizer is None:
        raise ValueError(
            "Resuming requires a restored optimizer."
        )

    model = model.to(device)

    x_train = x_train.to(device)
    y_train = y_train.to(device)

    # 2. Adam optimizer
    if optimizer is None:
        optimizer = optim.Adam(
            model.parameters(),
            lr=lr
        )

    # 3. Training history
    history_keys = [
        "loss",
        "mse",
        "lyap_weighted",
        "spectral_proxy"
    ]

    if history is None:
        history = {
            key: [] for key in history_keys
        }

    for key in history_keys:
        assert key in history
        assert len(history[key]) == start_epoch

    # 4. Full-batch training
    for ep in range(start_epoch, epochs):

        model.train()
        optimizer.zero_grad()

        # Forward + loss from Cell 3
        total_loss, pred_loss, lyap_loss, pred = (
            compute_ablation_loss(
                model,
                x_train,
                y_train,
                lyap_weight=lyap_weight
            )
        )

        if not torch.isfinite(total_loss).item():
            raise FloatingPointError(
                f"Non-finite loss at epoch {ep}"
            )

        # Backpropagation
        total_loss.backward()

        # Update parameters
        optimizer.step()

        # 5. Record losses
        history["loss"].append(
            float(total_loss.detach().item())
        )

        history["mse"].append(
            float(pred_loss.detach().item())
        )

        history["lyap_weighted"].append(
            float(
                (lyap_weight * lyap_loss)
                .detach().item()
            )
        )

        # 6. Log the same spectral proxy as the author
        koop = (
            getattr(model, koop_attr, None)
            if koop_attr is not None
            else None
        )

        if isinstance(
            koop, StrictStableKoopmanOperator
        ):
            spectral_proxy = (
                torch.sigmoid(koop.S_raw)
                .detach()
                .abs()
                .max()
                .item()
            )
        else:
            spectral_proxy = 0.0

        history["spectral_proxy"].append(
            float(spectral_proxy)
        )

        # 7. Progress logging
        if verbose and (
            ep % max(1, epochs // 10) == 0
            or ep == epochs - 1
        ):
            print(
                f"Epoch {ep:>4}/{epochs} | "
                f"MSE={pred_loss.item():.6e} | "
                f"Lyap={(lyap_weight * lyap_loss).item():.6e} | "
                f"Total={total_loss.item():.6e}"
            )

        # 8. Optional checkpoint callback
        if (
            checkpoint_callback is not None
            and (
                (ep + 1) % checkpoint_interval == 0
                or ep == epochs - 1
            )
        ):
            checkpoint_callback(
                ep + 1,
                model,
                optimizer,
                history
            )

    # 9. Final Train predictions
    model.eval()

    with torch.no_grad():
        pred_train = (
            model(x_train)
            .detach()
            .cpu()
            .numpy()
        )

    # 10. Return all training information
    return {
        "model": model,
        "optimizer": optimizer,
        "pred_train": pred_train,
        "history": history,
        "epochs_completed": epochs
    }


# SMOKE TEST — ONLY 2 EPOCHS

# Use a small real data sample
SMOKE_DATASET = "CMIP6 Wind"
SMOKE_PATCH_LEN = 70
SMOKE_HORIZON = 8
SMOKE_EPOCHS = 2
SMOKE_SAMPLES = 16

set_seed(SEED)

x_train, y_train, _, _ = prepare_ablation_data(
    SMOKE_DATASET,
    SMOKE_PATCH_LEN,
    SMOKE_HORIZON
)

x_small = x_train[:SMOKE_SAMPLES]
y_small = y_train[:SMOKE_SAMPLES]

# Initialize Full DeepKoopFormer
smoke_model, smoke_koop_attr, smoke_lyap = (
    create_ablation_model(
        model_name="KoopPatchTST_Full",
        input_dim=x_small.shape[-1],
        patch_len=SMOKE_PATCH_LEN,
        horizon=SMOKE_HORIZON
    )
)

# Run 2 epochs
smoke_result = train_ablation_model(
    model=smoke_model,
    x_train=x_small,
    y_train=y_small,
    epochs=SMOKE_EPOCHS,
    lr=LEARNING_RATE,
    koop_attr=smoke_koop_attr,
    lyap_weight=smoke_lyap,
    device=DEVICE
)

# Validate results
smoke_history = smoke_result["history"]

assert smoke_result["epochs_completed"] == 2

assert smoke_result["pred_train"].shape == (
    SMOKE_SAMPLES,
    x_small.shape[-1] * SMOKE_HORIZON
)

assert all(
    len(values) == SMOKE_EPOCHS
    for values in smoke_history.values()
)

assert np.isfinite(
    smoke_result["pred_train"]
).all()

assert np.isfinite(
    smoke_history["loss"]
).all()

assert len(
    smoke_result["optimizer"].state
) > 0

# Final summary
print("\n=== TRAINING LOOP SMOKE TEST ===")
print("Dataset:", SMOKE_DATASET)
print("Model: KoopPatchTST_Full")
print("Device:", DEVICE)

print("Input:", tuple(x_small.shape))
print("Target:", tuple(y_small.shape))

print("\nEpochs completed:",
      smoke_result["epochs_completed"])

print("Initial loss:", smoke_history["loss"][0])
print("Final loss:", smoke_history["loss"][-1])

print("Prediction shape:",
      smoke_result["pred_train"].shape)

print("\nTraining Loop validated!")
print("Cell 4 completed successfully!")

# Clean up smoke-test GPU allocations
del smoke_result, smoke_model
del x_train, y_train, x_small, y_small
del smoke_history

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


Epoch    0/2 | MSE=2.358012e-01 | Lyap=0.000000e+00 | Total=2.358012e-01
Epoch    1/2 | MSE=1.858958e-01 | Lyap=0.000000e+00 | Total=1.858958e-01

=== TRAINING LOOP SMOKE TEST ===
Dataset: CMIP6 Wind
Model: KoopPatchTST_Full
Device: cuda
Input: (16, 70, 5)
Target: (16, 40)

Epochs completed: 2
Initial loss: 0.23580117523670197
Final loss: 0.18589583039283752
Prediction shape: (16, 40)

Training Loop validated!
Cell 4 completed successfully!


TRAIN/TEST EVALUATION

In [5]:
# 1. Regression metrics
def calculate_regression_metrics(y_true, y_pred):

    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    assert y_true.shape == y_pred.shape
    assert np.isfinite(y_true).all()
    assert np.isfinite(y_pred).all()

    errors = y_pred - y_true

    mse = float(np.mean(errors ** 2))
    mae = float(np.mean(np.abs(errors)))

    return {
        "MSE": mse,
        "MAE": mae
    }


# 2. Batched prediction (inference only)
@torch.no_grad()
def predict_ablation_model(
    model,
    x,
    device=DEVICE,
    batch_size=128
):
    if batch_size <= 0:
        raise ValueError("batch_size must be positive.")

    if len(x) == 0:
        raise ValueError("Cannot predict an empty dataset.")

    model = model.to(device)

    was_training = model.training
    model.eval()

    predictions = []

    try:
        for start in range(0, len(x), batch_size):

            x_batch = x[
                start:start + batch_size
            ].to(device)

            pred_batch = model(x_batch)

            predictions.append(
                pred_batch.detach().cpu().numpy()
            )

    finally:
        model.train(was_training)

    return np.concatenate(
        predictions, axis=0
    )


# 3. Unified Train/Test evaluation
def evaluate_ablation_model(
    model,
    x_train,
    y_train,
    x_test,
    y_test,
    device=DEVICE,
    batch_size=128
):
    """
    Evaluate one trained model on Train and Test.

    Returns metrics, predictions and ground truth.
    Does not update model parameters.
    """

    results = {}

    for split_name, x, y in [
        ("Train", x_train, y_train),
        ("Test", x_test, y_test)
    ]:

        pred = predict_ablation_model(
            model,
            x,
            device=device,
            batch_size=batch_size
        )

        truth = y.detach().cpu().numpy()

        assert pred.shape == truth.shape
        assert pred.shape[0] == len(x)

        metrics = calculate_regression_metrics(
            truth,
            pred
        )

        results[split_name] = {
            "MSE": metrics["MSE"],
            "MAE": metrics["MAE"],
            "Samples": len(x),
            "predictions": pred,
            "ground_truth": truth
        }

    return results


# EVALUATION SMOKE TEST — NO TRAINING

# 4. Test configuration
EVAL_DATASET = "CMIP6 Wind"
EVAL_PATCH_LEN = 70
EVAL_HORIZON = 8

set_seed(SEED)

x_train, y_train, x_test, y_test = (
    prepare_ablation_data(
        EVAL_DATASET,
        EVAL_PATCH_LEN,
        EVAL_HORIZON
    )
)

# Small subsets for a quick evaluation test
x_train_small = x_train[:32]
y_train_small = y_train[:32]

x_test_small = x_test[:12]
y_test_small = y_test[:12]

# 5. Validate metric formulas with known values
known_true = np.array([0.0, 2.0])
known_pred = np.array([1.0, 2.0])

known_metrics = calculate_regression_metrics(
    known_true, known_pred
)

assert np.isclose(known_metrics["MSE"], 0.5)
assert np.isclose(known_metrics["MAE"], 0.5)

# 6. Test both a baseline and a Koopman model
EVAL_MODELS = [
    "DLinear",
    "KoopPatchTST_Full"
]

evaluation_records = []

for model_name in EVAL_MODELS:

    model, _, _ = create_ablation_model(
        model_name,
        input_dim=5,
        patch_len=EVAL_PATCH_LEN,
        horizon=EVAL_HORIZON
    )

    # First evaluation: batch size 7
    result = evaluate_ablation_model(
        model,
        x_train_small,
        y_train_small,
        x_test_small,
        y_test_small,
        device=DEVICE,
        batch_size=7
    )

    # Second evaluation: different batch size
    reference = evaluate_ablation_model(
        model,
        x_train_small,
        y_train_small,
        x_test_small,
        y_test_small,
        device=DEVICE,
        batch_size=32
    )

    for split_name in ["Train", "Test"]:

        current = result[split_name]
        expected = reference[split_name]

        # Predictions should not depend on inference batch size
        np.testing.assert_allclose(
            current["predictions"],
            expected["predictions"],
            rtol=1e-4,
            atol=1e-5
        )

        # Validate metrics independently
        metrics = calculate_regression_metrics(
            current["ground_truth"],
            current["predictions"]
        )

        assert np.isclose(
            current["MSE"], metrics["MSE"]
        )

        assert np.isclose(
            current["MAE"], metrics["MAE"]
        )

        assert current["MSE"] >= 0
        assert current["MAE"] >= 0

        evaluation_records.append({
            "Model": model_name,
            "Set": split_name,
            "Samples": current["Samples"],
            "MSE": current["MSE"],
            "MAE": current["MAE"],
            "Status": "PASSED"
        })

    del model


# 7. Validation summary
evaluation_df = pd.DataFrame(
    evaluation_records
)

assert len(evaluation_df) == 4
assert (evaluation_df["Status"] == "PASSED").all()

print("=== ABLATION EVALUATION VALIDATION ===")
print("Dataset:", EVAL_DATASET)
print("Patch length:", EVAL_PATCH_LEN)
print("Horizon:", EVAL_HORIZON)
print("Device:", DEVICE)

print("\n=== TRAIN/TEST EVALUATION ===")
display(evaluation_df)

print("\n=== VALIDATION SUMMARY ===")
print("Models tested:", len(EVAL_MODELS))
print("Train/Test evaluations:", len(evaluation_df))

print("\nMSE/MAE formulas validated.")
print("Inference batch-size consistency validated.")
print("Train and Test predictions validated.")

print("\nNOTE: Metrics are from UNTRAINED models.")
print("These are NOT official ablation results.")

print("\nCell 5 completed successfully!")


=== ABLATION EVALUATION VALIDATION ===
Dataset: CMIP6 Wind
Patch length: 70
Horizon: 8
Device: cuda

=== TRAIN/TEST EVALUATION ===


,Model,Set,Samples,MSE,MAE,Status
0,DLinear,Train,32,0.241567,0.436626,PASSED
1,DLinear,Test,12,0.177995,0.371339,PASSED
2,KoopPatchTST_Full,Train,32,0.267662,0.419416,PASSED
3,KoopPatchTST_Full,Test,12,0.198202,0.353796,PASSED



=== VALIDATION SUMMARY ===
Models tested: 2
Train/Test evaluations: 4

MSE/MAE formulas validated.
Inference batch-size consistency validated.
Train and Test predictions validated.

NOTE: Metrics are from UNTRAINED models.
These are NOT official ablation results.

Cell 5 completed successfully!


CHECKPOINT & RESUME TRAINING

In [6]:
import copy

# 1. Convert nested tensors to CPU for portable checkpoints
def to_cpu_copy(obj):

    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().clone()

    if isinstance(obj, dict):
        return {
            k: to_cpu_copy(v)
            for k, v in obj.items()
        }

    if isinstance(obj, list):
        return [to_cpu_copy(v) for v in obj]

    if isinstance(obj, tuple):
        return tuple(to_cpu_copy(v) for v in obj)

    return copy.deepcopy(obj)


# 2. Save full training state
def save_ablation_checkpoint(
    epoch, model, optimizer, history,
    path, config
):

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    # Store NumPy RNG in primitive Python types
    np_state = np.random.get_state()

    checkpoint = {
        "version": 1,
        "epoch": int(epoch),
        "config": dict(config),

        "model_state_dict": to_cpu_copy(
            model.state_dict()
        ),

        "optimizer_state_dict": to_cpu_copy(
            optimizer.state_dict()
        ),

        "history": copy.deepcopy(history),

        "rng_torch": torch.get_rng_state(),
        "rng_cuda": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else []
        ),

        "rng_numpy": {
            "name": np_state[0],
            "keys": np_state[1].tolist(),
            "pos": int(np_state[2]),
            "has_gauss": int(np_state[3]),
            "cached_gaussian": float(np_state[4])
        },

        "rng_python": random.getstate()
    }

    # Atomic replacement: prevent partial checkpoint files
    temp_path = path.with_name(
        path.name + ".tmp"
    )

    torch.save(checkpoint, temp_path)
    temp_path.replace(path)


# 3. Restore training state
def load_ablation_checkpoint(
    path,
    expected_config,
    device=DEVICE
):

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    if checkpoint["config"] != expected_config:
        raise ValueError(
            "Checkpoint configuration does not match "
            "the requested experiment."
        )

    config = checkpoint["config"]

    # Recreate the correct model architecture
    model, koop_attr, lyap_weight = (
        create_ablation_model(
            model_name=config["model_name"],
            input_dim=config["input_dim"],
            patch_len=config["patch_len"],
            horizon=config["horizon"]
        )
    )

    model = model.to(device)

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    # Restore Adam optimizer
    optimizer = optim.Adam(
        model.parameters(),
        lr=config["lr"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    # Restore history and epoch
    history = checkpoint["history"]
    start_epoch = checkpoint["epoch"]

    assert all(
        len(values) == start_epoch
        for values in history.values()
    )

    # Restore RNG AFTER creating the model and optimizer
    torch.set_rng_state(
        checkpoint["rng_torch"]
    )

    if device.type == "cuda":
        torch.cuda.set_rng_state_all(
            checkpoint["rng_cuda"]
        )

    saved_np = checkpoint["rng_numpy"]

    np.random.set_state((
        saved_np["name"],
        np.asarray(
            saved_np["keys"], dtype=np.uint32
        ),
        saved_np["pos"],
        saved_np["has_gauss"],
        saved_np["cached_gaussian"]
    ))

    random.setstate(
        checkpoint["rng_python"]
    )

    return (
        model,
        optimizer,
        history,
        start_epoch,
        koop_attr,
        lyap_weight
    )


# ============================================================
# SMOKE TEST — COMPARE CONTINUOUS VS RESUMED TRAINING
# ============================================================

# 4. Small test configuration
RESUME_DATASET = "CMIP6 Wind"
RESUME_MODEL = "KoopPatchTST_Full"
RESUME_PATCH_LEN = 70
RESUME_HORIZON = 8

RESUME_TOTAL_EPOCHS = 4
RESUME_SPLIT_EPOCH = 2

x_train, y_train, _, _ = prepare_ablation_data(
    RESUME_DATASET,
    RESUME_PATCH_LEN,
    RESUME_HORIZON
)

x_small = x_train[:8]
y_small = y_train[:8]

run_config = {
    "dataset_name": RESUME_DATASET,
    "model_name": RESUME_MODEL,
    "input_dim": 5,
    "patch_len": RESUME_PATCH_LEN,
    "horizon": RESUME_HORIZON,
    "epochs": RESUME_TOTAL_EPOCHS,
    "lr": LEARNING_RATE,
    "lyap_weight": LYAP_WEIGHT,
    "data_sha256": ABLATION_DATASETS[
        RESUME_DATASET
    ]["raw_sha256"]
}

checkpoint_path = (
    RESULTS_DIR / "_checkpoint_smoke_test.pt"
)

# 5. Reference run: uninterrupted 4 epochs
set_seed(SEED)

reference_model, reference_attr, reference_lyap = (
    create_ablation_model(
        RESUME_MODEL, 5,
        RESUME_PATCH_LEN,
        RESUME_HORIZON
    )
)

reference = train_ablation_model(
    model=reference_model,
    x_train=x_small,
    y_train=y_small,
    epochs=RESUME_TOTAL_EPOCHS,
    lr=LEARNING_RATE,
    koop_attr=reference_attr,
    lyap_weight=reference_lyap,
    verbose=False
)

# 6. Interrupted run: first 2 epochs
set_seed(SEED)

interrupted_model, interrupted_attr, interrupted_lyap = (
    create_ablation_model(
        RESUME_MODEL, 5,
        RESUME_PATCH_LEN,
        RESUME_HORIZON
    )
)

interrupted = train_ablation_model(
    model=interrupted_model,
    x_train=x_small,
    y_train=y_small,
    epochs=RESUME_SPLIT_EPOCH,
    lr=LEARNING_RATE,
    koop_attr=interrupted_attr,
    lyap_weight=interrupted_lyap,
    checkpoint_callback=lambda ep, model, opt, hist: (
        save_ablation_checkpoint(
            ep, model, opt, hist,
            checkpoint_path, run_config
        )
    ),
    checkpoint_interval=RESUME_SPLIT_EPOCH,
    verbose=False
)

assert checkpoint_path.is_file()

# 7. Restore from checkpoint
(
    resumed_model,
    resumed_optimizer,
    resumed_history,
    resumed_epoch,
    resumed_attr,
    resumed_lyap
) = load_ablation_checkpoint(
    checkpoint_path,
    expected_config=run_config
)

assert resumed_epoch == 2

# 8. Continue from epoch 2 to epoch 4
resumed = train_ablation_model(
    model=resumed_model,
    x_train=x_small,
    y_train=y_small,
    epochs=RESUME_TOTAL_EPOCHS,
    lr=LEARNING_RATE,
    koop_attr=resumed_attr,
    lyap_weight=resumed_lyap,
    optimizer=resumed_optimizer,
    start_epoch=resumed_epoch,
    history=resumed_history,
    verbose=False
)

# 9. Compare training histories
reference_losses = np.asarray(
    reference["history"]["loss"]
)

resumed_losses = np.asarray(
    resumed["history"]["loss"]
)

np.testing.assert_allclose(
    reference_losses,
    resumed_losses,
    rtol=1e-4,
    atol=1e-5
)

# 10. Compare final model parameters
max_param_diff = 0.0

for name, ref_param in (
    reference["model"].state_dict().items()
):

    resumed_param = (
        resumed["model"].state_dict()[name]
    )

    torch.testing.assert_close(
        ref_param,
        resumed_param,
        rtol=1e-4,
        atol=1e-5
    )

    diff = (
        ref_param.detach().cpu()
        - resumed_param.detach().cpu()
    ).abs().max().item()

    max_param_diff = max(
        max_param_diff, diff
    )

# 11. Compare final predictions
np.testing.assert_allclose(
    reference["pred_train"],
    resumed["pred_train"],
    rtol=1e-4,
    atol=1e-5
)

# 12. Summary
print("=== CHECKPOINT & RESUME VALIDATION ===")

print("Dataset:", RESUME_DATASET)
print("Model:", RESUME_MODEL)

print("\nReference epochs:", RESUME_TOTAL_EPOCHS)
print("Interrupted at epoch:", RESUME_SPLIT_EPOCH)
print("Resumed from epoch:", resumed_epoch)
print("Final epoch:", resumed["epochs_completed"])

print("\nReference losses:", reference_losses)
print("Resumed losses:  ", resumed_losses)

print("\nMaximum parameter difference:",
      max_param_diff)

print("Checkpoint size (MB):",
      round(
          checkpoint_path.stat().st_size / 1024**2,
          3
      )
)

print("\nModel state restored.")
print("Optimizer state restored.")
print("Training history restored.")
print("Random generator states restored.")

print("\nContinuous and resumed training matched!")
print("Cell 6 completed successfully!")

# 13. Remove temporary smoke-test checkpoint
checkpoint_path.unlink()

del reference, interrupted, resumed
del reference_model, interrupted_model, resumed_model
del resumed_optimizer, resumed_history
del x_train, y_train, x_small, y_small

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


=== CHECKPOINT & RESUME VALIDATION ===
Dataset: CMIP6 Wind
Model: KoopPatchTST_Full

Reference epochs: 4
Interrupted at epoch: 2
Resumed from epoch: 2
Final epoch: 4

Reference losses: [0.16848944 0.12456656 0.09918673 0.07453298]
Resumed losses:   [0.16848944 0.12456656 0.09918673 0.07453298]

Maximum parameter difference: 0.0
Checkpoint size (MB): 2.639

Model state restored.
Optimizer state restored.
Training history restored.
Random generator states restored.

Continuous and resumed training matched!
Cell 6 completed successfully!


END-TO-END EXPERIMENT RUNNER + SHORT SMOKE TEST

In [7]:
import hashlib
import json
import re
import time
from pathlib import Path


def _safe_slug(value):
    return re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_")


def _processed_data_hash(bundle):
    """Fingerprint the actual scaled train/test data, not only the raw file."""
    digest = hashlib.sha256()
    for split in ("train", "test"):
        arr = bundle[split].detach().cpu().contiguous().numpy()
        digest.update(split.encode("utf-8"))
        digest.update(str(arr.shape).encode("utf-8"))
        digest.update(arr.tobytes())
    return digest.hexdigest()


def run_one_ablation(
    dataset_name,
    model_name,
    patch_len,
    horizon,
    epochs=EPOCHS,
    seed=SEED,
    checkpoint_every=500,
    save_predictions=False,
    verbose=False,
    force_restart=False,
):
    """
    Train, evaluate, persist, and (if interrupted) resume one configuration.
    A completed metrics.json is the source of truth for skip-on-rerun.
    """
    if dataset_name not in ABLATION_DATASETS:
        raise ValueError(f"Unknown dataset: {dataset_name}")
    if model_name not in MODEL_VARIANTS:
        raise ValueError(f"Unknown model: {model_name}")
    if patch_len not in PATCH_LENGTHS or horizon not in HORIZONS:
        raise ValueError(f"Invalid patch/horizon: {patch_len}, {horizon}")
    if epochs <= 0 or checkpoint_every <= 0:
        raise ValueError("epochs and checkpoint_every must be positive")

    bundle = ABLATION_DATASETS[dataset_name]
    run_dir = (
        RESULTS_DIR / "ablation_runs" / _safe_slug(dataset_name)
        / f"p{patch_len:03d}_h{horizon:02d}"
        / _safe_slug(model_name) / f"epochs{epochs}_seed{seed}"
    )
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = run_dir / "latest.pt"
    metrics_path = run_dir / "metrics.json"
    history_path = run_dir / "training_history.csv"
    prediction_path = run_dir / "predictions.npz"

    # The Lyapunov coefficient is tied to the variant, not user-provided.
    expected_lyap_weight = (
        float(LYAP_WEIGHT) if model_name == "KoopPatchTST_Full" else 0.0
    )
    config = {
        "dataset_name": dataset_name,
        "model_name": model_name,
        "input_dim": int(bundle["input_dim"]),
        "patch_len": int(patch_len),
        "horizon": int(horizon),
        "epochs": int(epochs),
        "lr": float(LEARNING_RATE),
        "lyap_weight": expected_lyap_weight,
        "seed": int(seed),
        "raw_sha256": str(bundle["raw_sha256"]),
        "processed_sha256": _processed_data_hash(bundle),
        "d_model": int(D_MODEL),
        "lstm_hidden_dim": int(LSTM_HIDDEN_DIM),
        "lstm_num_layers": int(LSTM_NUM_LAYERS),
        "rho_max": float(RHO_MAX),
    }

    if force_restart:
        for file_path in (checkpoint_path, metrics_path, history_path, prediction_path):
            file_path.unlink(missing_ok=True)

    if metrics_path.exists():
        with metrics_path.open("r", encoding="utf-8") as stream:
            previous = json.load(stream)
        if previous["config"] != config or previous.get("status") != "complete":
            raise ValueError(f"Existing metrics are incompatible: {metrics_path}")
        if not checkpoint_path.exists():
            raise FileNotFoundError(
                f"Metrics exist but final checkpoint is missing: {checkpoint_path}"
            )
        print(f"[SKIP — completed] {dataset_name} | {model_name} | P={patch_len}, H={horizon}")
        return previous

    # Do not leak test observations into training.
    x_train, y_train, x_test, y_test = prepare_ablation_data(
        dataset_name, patch_len, horizon
    )

    # Initializing then restoring in this order is intentional: loader restores RNG.
    set_seed(seed)
    if checkpoint_path.exists():
        model, optimizer, history, start_epoch, koop_attr, lyap_weight = (
            load_ablation_checkpoint(checkpoint_path, expected_config=config, device=DEVICE)
        )
        print(f"[RESUME] {model_name}: epoch {start_epoch}/{epochs}")
    else:
        model, koop_attr, lyap_weight = create_ablation_model(
            model_name, int(bundle["input_dim"]), patch_len, horizon
        )
        optimizer, history, start_epoch = None, None, 0
        print(f"[START]  {dataset_name} | {model_name} | P={patch_len}, H={horizon}")

    assert lyap_weight == expected_lyap_weight
    if not (0 <= start_epoch <= epochs):
        raise ValueError(f"Invalid checkpoint epoch: {start_epoch}/{epochs}")

    def save_callback(epoch, current_model, current_optimizer, current_history):
        save_ablation_checkpoint(
            epoch=epoch,
            model=current_model,
            optimizer=current_optimizer,
            history=current_history,
            path=checkpoint_path,
            config=config,
        )

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    start_time = time.perf_counter()

    if start_epoch < epochs:
        trained = train_ablation_model(
            model=model,
            x_train=x_train,
            y_train=y_train,
            epochs=epochs,
            lr=LEARNING_RATE,
            koop_attr=koop_attr,
            lyap_weight=lyap_weight,
            device=DEVICE,
            optimizer=optimizer,
            start_epoch=start_epoch,
            history=history,
            checkpoint_callback=save_callback,
            checkpoint_interval=checkpoint_every,
            verbose=verbose,
        )
        model = trained["model"]
        optimizer = trained["optimizer"]
        history = trained["history"]
        del trained
    else:
        model.eval()

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    training_seconds_this_session = time.perf_counter() - start_time

    eval_start = time.perf_counter()
    evaluation = evaluate_ablation_model(
        model, x_train, y_train, x_test, y_test,
        device=DEVICE, batch_size=128
    )
    evaluation_seconds = time.perf_counter() - eval_start

    history_df = pd.DataFrame(history)
    if len(history_df) != epochs:
        raise RuntimeError(f"Unexpected history length: {len(history_df)} != {epochs}")
    history_df.insert(0, "epoch", np.arange(1, epochs + 1))
    history_df.to_csv(history_path, index=False)

    if save_predictions:
        np.savez_compressed(
            prediction_path,
            train_pred=evaluation["Train"]["predictions"],
            train_true=evaluation["Train"]["ground_truth"],
            test_pred=evaluation["Test"]["predictions"],
            test_true=evaluation["Test"]["ground_truth"],
        )

    peak_vram_gb = (
        float(torch.cuda.max_memory_allocated() / (1024 ** 3))
        if DEVICE.type == "cuda" else None
    )
    record = {
        "status": "complete",
        "config": config,
        "dataset_status": str(bundle["status"]),
        "train_samples": int(len(x_train)),
        "test_samples": int(len(x_test)),
        "train_mse": float(evaluation["Train"]["MSE"]),
        "train_mae": float(evaluation["Train"]["MAE"]),
        "test_mse": float(evaluation["Test"]["MSE"]),
        "test_mae": float(evaluation["Test"]["MAE"]),
        "first_training_loss": float(history["loss"][0]),
        "last_training_loss": float(history["loss"][-1]),
        "last_weighted_lyap": float(history["lyap_weighted"][-1]),
        "training_seconds_this_session": float(training_seconds_this_session),
        "evaluation_seconds": float(evaluation_seconds),
        "peak_pytorch_vram_gb": peak_vram_gb,
        "checkpoint": str(checkpoint_path),
    }

    # Save metrics LAST, so a completed metrics file means every artifact exists.
    temp_path = metrics_path.with_name(metrics_path.name + ".tmp")
    with temp_path.open("w", encoding="utf-8") as stream:
        json.dump(record, stream, indent=2, ensure_ascii=False, allow_nan=False)
    temp_path.replace(metrics_path)

    print(
        f"[DONE] {dataset_name} | {model_name} | P={patch_len}, H={horizon} | "
        f"Test MSE={record['test_mse']:.6f} | MAE={record['test_mae']:.6f} | "
        f"session={training_seconds_this_session:.1f}s"
    )

    del model, optimizer, evaluation
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return record


# ----------------------------------
# Short smoke test: just 3 epochs.
# This is NOT an official 4000-epoch result.
# ----------------------------------
SMOKE_3EP_RESULT = run_one_ablation(
    dataset_name="CMIP6 Wind",
    model_name="KoopPatchTST_Full",
    patch_len=120,
    horizon=8,
    epochs=3,
    checkpoint_every=2,
    save_predictions=True,
    verbose=True,
)

assert SMOKE_3EP_RESULT["status"] == "complete"
assert SMOKE_3EP_RESULT["train_samples"] == 2208
assert SMOKE_3EP_RESULT["test_samples"] == 456
assert Path(SMOKE_3EP_RESULT["checkpoint"]).is_file()
print("Cell 7 completed successfully — end-to-end runner ready.")


[SKIP — completed] CMIP6 Wind | KoopPatchTST_Full | P=120, H=8
Cell 7 completed successfully — end-to-end runner ready.


GRID PLAN AND RESUMABLE EXECUTION

In [8]:
from itertools import product


def build_ablation_grid():
    rows = []
    for dataset_name, patch_len, horizon, model_name in product(
        ABLATION_DATASETS.keys(), PATCH_LENGTHS, HORIZONS, MODEL_VARIANTS
    ):
        bundle = ABLATION_DATASETS[dataset_name]
        n_train = len(bundle["train"]) - patch_len - horizon
        n_test = len(bundle["test"]) - patch_len - horizon
        if min(n_train, n_test) <= 0:
            raise ValueError(f"Insufficient windows: {dataset_name} P={patch_len} H={horizon}")
        rows.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "P": int(patch_len),
            "H": int(horizon),
            "TrainWindows": int(n_train),
            "TestWindows": int(n_test),
            "Epochs": int(EPOCHS),
            "Seed": int(SEED),
        })
    result = pd.DataFrame(rows)
    expected = (
        len(ABLATION_DATASETS) * len(PATCH_LENGTHS)
        * len(HORIZONS) * len(MODEL_VARIANTS)
    )
    assert len(result) == expected == 1344
    assert not result.duplicated(["Dataset", "Model", "P", "H"]).any()
    return result


ABLATION_GRID = build_ablation_grid()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ABLATION_GRID.to_csv(RESULTS_DIR / "ablation_grid_plan.csv", index=False)

print("=== ABLATION GRID PLAN ===")
print("Datasets:", ABLATION_GRID["Dataset"].nunique())
print("Models:", ABLATION_GRID["Model"].nunique())
print("Patch lengths:", ABLATION_GRID["P"].nunique())
print("Horizons:", ABLATION_GRID["H"].nunique())
print("Total configurations:", len(ABLATION_GRID))
display(ABLATION_GRID.head(12))

# IMPORTANT: leave False on your laptop.
# Switch to True only when you are ready to execute the complete grid.
RUN_FULL_GRID = False

# Optional filters on the rented GPU: None means ALL.
ONLY_DATASETS = None      # e.g. ["CMIP6 Wind"]
ONLY_MODELS = None        # e.g. ["KoopPatchTST_Full"]
MAX_RUNS = None           # e.g. 2 for a short real trial


selected_grid = ABLATION_GRID.copy()
if ONLY_DATASETS is not None:
    selected_grid = selected_grid[selected_grid["Dataset"].isin(ONLY_DATASETS)]
if ONLY_MODELS is not None:
    selected_grid = selected_grid[selected_grid["Model"].isin(ONLY_MODELS)]
if MAX_RUNS is not None:
    selected_grid = selected_grid.head(MAX_RUNS)

print("Selected configurations:", len(selected_grid))

if not RUN_FULL_GRID:
    print("DRY RUN ONLY — no 4000-epoch training started.")
    print("Cell 8 completed successfully — grid is ready for the rented GPU.")
else:
    if len(selected_grid) == 0:
        raise ValueError("No configurations selected.")

    for job_index, row in enumerate(selected_grid.itertuples(index=False), start=1):
        print(
            f"\n[{job_index}/{len(selected_grid)}] "
            f"{row.Dataset} | {row.Model} | P={row.P}, H={row.H}"
        )
        # An existing completed metrics file is skipped.
        # An existing checkpoint without metrics is resumed.
        # Exceptions stop the grid so that errors are investigated.
        run_one_ablation(
            dataset_name=row.Dataset,
            model_name=row.Model,
            patch_len=int(row.P),
            horizon=int(row.H),
            epochs=EPOCHS,
            seed=SEED,
            checkpoint_every=500,
            save_predictions=False,
            verbose=False,
        )

    print("\nSelected ablation grid finished.")
    print("Run Cell 9 to aggregate completed metrics.")


=== ABLATION GRID PLAN ===
Datasets: 4
Models: 6
Patch lengths: 7
Horizons: 8
Total configurations: 1344


,Dataset,Model,P,H,TrainWindows,TestWindows,Epochs,Seed
0,Electricity,DLinear,70,2,2728,628,4000,7
1,Electricity,LSTM,70,2,2728,628,4000,7
2,Electricity,PatchTST,70,2,2728,628,4000,7
3,Electricity,PatchTST+LinearLatent,70,2,2728,628,4000,7
4,Electricity,KoopPatchTST_NoLyap,70,2,2728,628,4000,7
5,Electricity,KoopPatchTST_Full,70,2,2728,628,4000,7
6,Electricity,DLinear,70,4,2726,626,4000,7
7,Electricity,LSTM,70,4,2726,626,4000,7
8,Electricity,PatchTST,70,4,2726,626,4000,7
9,Electricity,PatchTST+LinearLatent,70,4,2726,626,4000,7


Selected configurations: 1344
DRY RUN ONLY — no 4000-epoch training started.
Cell 8 completed successfully — grid is ready for the rented GPU.


GRID PLAN AND RESUMABLE EXECUTION

In [9]:
from itertools import product


def build_ablation_grid():
    rows = []
    for dataset_name, patch_len, horizon, model_name in product(
        ABLATION_DATASETS.keys(), PATCH_LENGTHS, HORIZONS, MODEL_VARIANTS
    ):
        bundle = ABLATION_DATASETS[dataset_name]
        n_train = len(bundle["train"]) - patch_len - horizon
        n_test = len(bundle["test"]) - patch_len - horizon
        if min(n_train, n_test) <= 0:
            raise ValueError(f"Insufficient windows: {dataset_name} P={patch_len} H={horizon}")
        rows.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "P": int(patch_len),
            "H": int(horizon),
            "TrainWindows": int(n_train),
            "TestWindows": int(n_test),
            "Epochs": int(EPOCHS),
            "Seed": int(SEED),
        })
    result = pd.DataFrame(rows)
    expected = (
        len(ABLATION_DATASETS) * len(PATCH_LENGTHS)
        * len(HORIZONS) * len(MODEL_VARIANTS)
    )
    assert len(result) == expected == 1344
    assert not result.duplicated(["Dataset", "Model", "P", "H"]).any()
    return result


ABLATION_GRID = build_ablation_grid()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ABLATION_GRID.to_csv(RESULTS_DIR / "ablation_grid_plan.csv", index=False)

print("=== ABLATION GRID PLAN ===")
print("Datasets:", ABLATION_GRID["Dataset"].nunique())
print("Models:", ABLATION_GRID["Model"].nunique())
print("Patch lengths:", ABLATION_GRID["P"].nunique())
print("Horizons:", ABLATION_GRID["H"].nunique())
print("Total configurations:", len(ABLATION_GRID))
display(ABLATION_GRID.head(12))

# IMPORTANT: leave False on your laptop.
# Switch to True only when you are ready to execute the complete grid.
RUN_FULL_GRID = False

# Optional filters on the rented GPU: None means ALL.
ONLY_DATASETS = None      # e.g. ["CMIP6 Wind"]
ONLY_MODELS = None        # e.g. ["KoopPatchTST_Full"]
MAX_RUNS = None           # e.g. 2 for a short real trial


selected_grid = ABLATION_GRID.copy()
if ONLY_DATASETS is not None:
    selected_grid = selected_grid[selected_grid["Dataset"].isin(ONLY_DATASETS)]
if ONLY_MODELS is not None:
    selected_grid = selected_grid[selected_grid["Model"].isin(ONLY_MODELS)]
if MAX_RUNS is not None:
    selected_grid = selected_grid.head(MAX_RUNS)

print("Selected configurations:", len(selected_grid))

if not RUN_FULL_GRID:
    print("DRY RUN ONLY — no 4000-epoch training started.")
    print("Cell 8 completed successfully — grid is ready for the rented GPU.")
else:
    if len(selected_grid) == 0:
        raise ValueError("No configurations selected.")

    for job_index, row in enumerate(selected_grid.itertuples(index=False), start=1):
        print(
            f"\n[{job_index}/{len(selected_grid)}] "
            f"{row.Dataset} | {row.Model} | P={row.P}, H={row.H}"
        )
        # An existing completed metrics file is skipped.
        # An existing checkpoint without metrics is resumed.
        # Exceptions stop the grid so that errors are investigated.
        run_one_ablation(
            dataset_name=row.Dataset,
            model_name=row.Model,
            patch_len=int(row.P),
            horizon=int(row.H),
            epochs=EPOCHS,
            seed=SEED,
            checkpoint_every=500,
            save_predictions=False,
            verbose=False,
        )

    print("\nSelected ablation grid finished.")
    print("Run Cell 9 to aggregate completed metrics.")


=== ABLATION GRID PLAN ===
Datasets: 4
Models: 6
Patch lengths: 7
Horizons: 8
Total configurations: 1344


,Dataset,Model,P,H,TrainWindows,TestWindows,Epochs,Seed
0,Electricity,DLinear,70,2,2728,628,4000,7
1,Electricity,LSTM,70,2,2728,628,4000,7
2,Electricity,PatchTST,70,2,2728,628,4000,7
3,Electricity,PatchTST+LinearLatent,70,2,2728,628,4000,7
4,Electricity,KoopPatchTST_NoLyap,70,2,2728,628,4000,7
5,Electricity,KoopPatchTST_Full,70,2,2728,628,4000,7
6,Electricity,DLinear,70,4,2726,626,4000,7
7,Electricity,LSTM,70,4,2726,626,4000,7
8,Electricity,PatchTST,70,4,2726,626,4000,7
9,Electricity,PatchTST+LinearLatent,70,4,2726,626,4000,7


Selected configurations: 1344
DRY RUN ONLY — no 4000-epoch training started.
Cell 8 completed successfully — grid is ready for the rented GPU.


LOCAL ABLATION — DLINEAR + LSTM
448 configurations × 4000 epochs

In [ ]:

from pathlib import Path
import subprocess
import sys
import time
import torch

# ============================================================
# LOCAL ABLATION — DLINEAR + LSTM
# 448 configurations × 4000 epochs
# ============================================================

# 1. Locate project
project_root = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (
            p / "experiments" / "05_ablation"
            / "train_ablation_v2.py"
        ).is_file()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Cannot locate train_ablation_v2.py"
    )

exp_dir = project_root / "experiments" / "05_ablation"

script_path = exp_dir / "train_ablation_v2.py"
model_path = exp_dir / "02_ablation_model.ipynb"
data_dir = exp_dir / "data"
results_dir = exp_dir / "results"
runs_dir = results_dir / "ablation_runs"

results_dir.mkdir(parents=True, exist_ok=True)

# 2. Verify four datasets
required_files = [
    "electricity_ablation.pt",
    "financial_ablation.pt",
    "cmip6_wind_ablation.pt",
    "cmip6_pressure_candidate_2500.pt",
]

for filename in required_files:
    path = data_dir / filename
    assert path.is_file(), f"Missing: {path}"

assert model_path.is_file()

# 3. Verify local GPU
assert torch.cuda.is_available(), "CUDA unavailable"

print("=== LOCAL ABLATION ===")
print("Python:", sys.executable)
print("GPU:", torch.cuda.get_device_name(0))
print("Models: DLinear")
print("Configurations:", 4 * 7 * 8)
print("Datasets:", len(required_files))
print("Epochs:", 4000)
print("Results:", results_dir)

# 4. Build training command
command = [
    sys.executable,
    "-u",
    str(script_path),
    "--run",
    "--device", "cuda",
    "--models", "DLinear",
    "--epochs", "4000",
    "--checkpoint-every", "250",
    "--model-notebook", str(model_path),
    "--data-dir", str(data_dir),
    "--output-dir", str(runs_dir),
]

# 5. Save full console log
log_path = results_dir / "local_dlinear_only.log"

print("\nStarting training...")
print("Log:", log_path)

start_time = time.perf_counter()

with open(log_path, "a", encoding="utf-8") as log_file:

    log_file.write("\n=== NEW LOCAL TRAINING SESSION ===\n")
    log_file.flush()

    process = subprocess.Popen(
        command,
        cwd=str(exp_dir),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1
    )

    try:
        for line in process.stdout:

            # Save every log line
            log_file.write(line)
            log_file.flush()

            # Show important progress in notebook
            message = line.strip()

            show = (
                message.startswith((
                    "GPU:",
                    "Grid:",
                    "[",
                    "RESUME",
                    "SKIP",
                    "SAVED:",
                    "DONE",
                    "Completed configurations:",
                    "ALL REQUESTED",
                    "Traceback"
                ))
                or "epoch=4000/4000" in message
            )

            if show:
                print(message, flush=True)

        return_code = process.wait()

    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        print("Training interrupted. Saved checkpoints retained.")
        raise

elapsed = (time.perf_counter() - start_time) / 3600

print("\n=== LOCAL ABLATION SESSION ENDED ===")
print(f"Elapsed: {elapsed:.2f} hours")
print("Exit code:", return_code)
print("Log:", log_path)

if return_code != 0:
    raise RuntimeError(
        "Training stopped with an error. "
        f"Check {log_path}"
    )

print("All requested DLinear runs completed.")


=== LOCAL ABLATION ===
Python: c:\Users\Win 11 New\Desktop\pa gk\.venv\Scripts\python.exe
GPU: Quadro T1000
Models: DLinear
Configurations: 224
Datasets: 4
Epochs: 4000
Results: c:\Users\Win 11 New\Desktop\pa gk\experiments\05_ablation\results

Starting training...
Log: c:\Users\Win 11 New\Desktop\pa gk\experiments\05_ablation\results\local_dlinear_only.log
GPU: Quadro T1000
Grid: 224 configurations; epochs: 4000
[1/224] Electricity DLinear P=70 H=2
SKIP COMPLETED Electricity DLinear P=70 H=2
Completed configurations: 19; metrics rows: 38
[2/224] Electricity DLinear P=70 H=4
SKIP COMPLETED Electricity DLinear P=70 H=4
Completed configurations: 19; metrics rows: 38
[3/224] Electricity DLinear P=70 H=6
SKIP COMPLETED Electricity DLinear P=70 H=6
Completed configurations: 19; metrics rows: 38
[4/224] Electricity DLinear P=70 H=8
SKIP COMPLETED Electricity DLinear P=70 H=8
Completed configurations: 19; metrics rows: 38
[5/224] Electricity DLinear P=70 H=10
SKIP COMPLETED Electricity DLinea

In [14]:
from pathlib import Path

project_root = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (
            p / "experiments" / "05_ablation"
            / "train_ablation_v2.py"
        ).is_file()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Cannot locate train_ablation_v2.py"
    )
exp_dir = project_root / "experiments" / "05_ablation"
root = exp_dir / "results" / "ablation_runs"

for model in ["dlinear", "lstm"]:
    pattern = f"*/p*_h*/{model}/e4000_seed7"

    completed = list(root.glob(f"{pattern}/metrics.json"))
    checkpoints = list(root.glob(f"{pattern}/latest.pt"))

    print(f"\n{model.upper()}")
    print(f"Completed: {len(completed)}/224")
    print(f"Checkpoints: {len(checkpoints)}")
    print(f"Remaining: {224 - len(completed)}")


DLINEAR
Completed: 166/224
Checkpoints: 167
Remaining: 58

LSTM
Completed: 9/224
Checkpoints: 10
Remaining: 215


In [15]:

from pathlib import Path
import zipfile
import hashlib

# Locate experiment directory independently
exp_dir = next(
    (
        p
        for base in [Path.cwd(), *Path.cwd().parents]
        for p in [
            base,
            base / "experiments" / "05_ablation"
        ]
        if (p / "train_ablation_v2.py").is_file()
    ),
    None
)

if exp_dir is None:
    raise FileNotFoundError("Cannot locate 05_ablation")

data_names = [
    "electricity_ablation.pt",
    "financial_ablation.pt",
    "cmip6_wind_ablation.pt",
    "cmip6_pressure_candidate_2500.pt",
]

required = [
    exp_dir / "train_ablation_v2.py",
    exp_dir / "02_ablation_model.ipynb",
    *[exp_dir / "data" / f for f in data_names],
]

for path in required:
    assert path.is_file(), f"Missing: {path}"

runs = exp_dir / "results" / "ablation_runs"
assert runs.is_dir()

# Count saved local results
for model in ("dlinear", "lstm"):
    completed = list(
        runs.glob(
            f"*/p*_h*/{model}/e4000_seed7/metrics.json"
        )
    )
    checkpoints = list(
        runs.glob(
            f"*/p*_h*/{model}/e4000_seed7/latest.pt"
        )
    )
    print(
        f"{model}: {len(completed)} completed, "
        f"{len(checkpoints)} checkpoints"
    )

# Include all resume states
files = list(required)

files += [
    p for p in runs.rglob("*")
    if p.is_file()
]

# Include finished standalone model artifacts
for folder in (exp_dir / "results").glob(
    "*_grid_search_e4000"
):
    files += [
        p for p in folder.rglob("*.pt")
        if p.is_file()
    ]

files = sorted(set(files))

zip_path = (
    exp_dir.parent / "ablation_local_session3_upload.zip"
)

print(f"\nFiles to package: {len(files)}")
print("Creating:", zip_path)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1,
    allowZip64=True
) as zf:
    for i, path in enumerate(files, 1):
        zf.write(
            path,
            arcname=path.relative_to(exp_dir).as_posix()
        )
        if i % 100 == 0:
            print(f"Packed {i}/{len(files)}")

with zipfile.ZipFile(zip_path) as zf:
    names = set(zf.namelist())
    assert "train_ablation_v2.py" in names
    assert "02_ablation_model.ipynb" in names
    assert any(
        name.endswith("/metrics.json")
        for name in names
    )

print("\nPACKAGE READY")
print("ZIP:", zip_path)
print("Size (GB):", round(
    zip_path.stat().st_size / (1024 ** 3), 3
))
print(
    "Model SHA256:",
    hashlib.sha256(
        (exp_dir / "02_ablation_model.ipynb").read_bytes()
    ).hexdigest()
)


dlinear: 166 completed, 167 checkpoints
lstm: 9 completed, 10 checkpoints

Files to package: 713
Creating: c:\Users\Win 11 New\Desktop\pa gk\experiments\ablation_local_session3_upload.zip
Packed 100/713
Packed 200/713
Packed 300/713
Packed 400/713
Packed 500/713
Packed 600/713
Packed 700/713

PACKAGE READY
ZIP: c:\Users\Win 11 New\Desktop\pa gk\experiments\ablation_local_session3_upload.zip
Size (GB): 0.068
Model SHA256: 74b449a8ff1b6eb5a6ce0d25c272dc8491973f58aeb3f9295e44c59a4ab49bed
